# The lag of MAREA and of the other methods against the in-estuary tide gauges, at one site

**The question.** Inside an estuary the tide arrives late. MAREA reads that delay from the
imagery, one lag per 3-km band of distance from the mouth. How close is it, and how close are the
other ways of getting a lag (none, a global tide model, the other imagery methods, the
shallow-water celerity), to the lag the in-estuary tide gauges, the *judges*, show?

This is Stage L of validation v2 (`docs/validation_v2_spec.md` §6), run on the server on
2026-10-26 by `experiments/validation_v2.py`, written here as plain calls to `pyintertidal`, to
the benchmark's truth functions and to the method modules of `experiments/benchmark/lags/`.
Section 13 checks the numbers against the run's `lag_points.csv`;
`02b_lag_summary.ipynb` puts the sites together.

**How to read it**

* *Pre-lock, descriptive.* W3, the confirmatory window, stays blinded: no gauge is read there.
  Only the windows of the pre-lock table are scored: Y2023 at Escalda and Ems, Y2024 at Ferrol,
  W1, W2, Y2023 and Y2024 in the Wadden. Villaviciosa has no judge: its notebook shows the band
  lags and how they agree from one year to the next.
* *Catalogue instants.* A scene's instant is its catalogue (datatake-start) time. The tile is
  sensed 0–10.6 min later (median 6.5 min at the Ems), so every imagery lag is expected to read
  about that much low against the gauges. Nothing is corrected here: errors are attributed first.
* *Weak evidence.* The gauge lag is close to MAREA's own estimand, and MAREA's lag also absorbs
  wet/dry departures (ebb retention, drainage); the decisive test of the lag is the elevation
  against external heights (spec §6).

## Parameters

`SITE` picks the site; `run_02_on_server.sh` sets it through `NB_SITE`. Each site has the
windows of the pre-lock lag table; `NB_WINDOWS` (e.g. `W2`) shortens the list for a test.

In [ ]:
import os

SITE = os.environ.get("NB_SITE", "escalda")       # escalda | wadden | ems | ferrol | villaviciosa

# the production boundary (EOT20 blended with the n nearest IOC gauges, the in-estuary gauges held
# out; EOT20 alone at Villaviciosa) and the windows of the pre-lock lag table
SITES = {
    "escalda":      dict(n_gauges=2, held_out=["trnz"], windows=["Y2023"]),
    "wadden":       dict(n_gauges=1, held_out=["harl"], windows=["W1", "W2", "Y2023", "Y2024"]),
    "ems":          dict(n_gauges=1, held_out=["delf"], windows=["Y2023"]),
    "ferrol":       dict(n_gauges=1, held_out=["fer2"], windows=["Y2024"]),
    "villaviciosa": dict(n_gauges=None, held_out=[], windows=["Y2023", "Y2024", "W1"]),
}
S = SITES[SITE]
WINDOWS = os.environ["NB_WINDOWS"].split(",") if os.environ.get("NB_WINDOWS") else S["windows"]
assert "W3" not in WINDOWS, "W3 is the blinded confirmatory window: scoring it is the user's decision"

# [first day, day after the last) of each window (spec §2.7; W1 is the year 2025)
WINDOW_DATES = {"W1": ("2025-01-01", "2026-01-01"), "W2": ("2024-01-01", "2026-01-01"),
                "Y2023": ("2023-01-01", "2024-01-01"), "Y2024": ("2024-01-01", "2025-01-01")}

SETTING      = "C1.floor0.ndwi.D1"   # P0: SCL 4-7 clear, reflectance max(DN - offset, 0), NDWI, wet if > 0
GLOBAL_MODEL = "EOT20"               # the run's primary global model (FES2022 is not installed)
V2_ROOT = "results/validation_v2"    # the run: package (scenes, pixels), judge table, EMODnet tile
SCORES  = os.environ.get("NB_SCORES", f"{V2_ROOT}/scores")    # its lag_points.csv and lag_repro.csv
OUT_DIR = os.environ.get("NB_OUT", "results/lag_nb")

print(f"{SITE}: windows {WINDOWS} -> {OUT_DIR}/{SITE}.csv")

## 0 · Setup

The notebook runs from the repository root, so `pyintertidal`, `configs/` and the data are the
repository's. Two BLAS threads.

In [ ]:
import sys, time

for _ in range(3):                                # this notebook lives in notebooks/validation/
    if os.path.isdir("pyintertidal"):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "2")                 # before numpy loads its BLAS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

import pyintertidal as pit
from pyintertidal import geometry, marea
from pyintertidal import granadeiro2021 as g21
from pyintertidal import tide_estimators as te
from pyintertidal.boundary import make_boundary, PyTMDBoundary
from experiments.benchmark import judges, levels, package, readout, truth
from experiments.benchmark.fetch import emodnet
from experiments.benchmark.lags import aggregate, bt26, common, gm20, hd26, physics, ppl

os.makedirs(OUT_DIR, exist_ok=True)
print("repository", os.getcwd(), "· pyintertidal", pit.__version__)

## 1 · Study area and judges

The inputs are those of the run: its package of the site's 10 m Zarr stores (B03, B08 and SCL of
every scene, kept on the pixels that can be intertidal) and its judge table. A judge is an
in-estuary gauge; its *neighbourhood* is every pixel within 500 m (widened until it holds 100
intertidal pixels) and its distance from the mouth along the water, s_g, the median s of its 200
nearest intertidal pixels. A judge counts only where method-free rules pass: inside the frame,
more than 3 km from every gauge of the boundary, a gauge value near at least half of the scene
instants, and not dry.

In [ ]:
aoi = pit.sites.get(SITE)
pkg = package.V2Package(package.v2_site(SITE))    # the run's package of the 10 m Zarr stores
grid = pkg.grid                                   # pixel centres and CRS of the frame
H, W = pkg.shape
print(aoi)
print(pkg)

table = f"{V2_ROOT}/truth/{SITE}/judges.json"     # the run's judge table (neighbourhoods rebuilt on the grid)
JUDGES = judges.load(table, grid) if os.path.exists(table) else []
pd.DataFrame([{"judge": J["code"], "name": J["name"], "provider": J["provider"],
               "s_g (km)": round(J["s_g_km"], 1), "mouth (s < 3 km)": J["mouth"],
               "in frame": J["in_frame"], "nearest boundary gauge (km)": J["nearest_member_km"],
               "flags": ", ".join(J["flags"])} for J in JUDGES])

In [ ]:
s_union = np.asarray(pkg.geometry["s_frame"], float).reshape(H, W)   # s over the whole record, km
fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(s_union, cmap="viridis", interpolation="nearest")
for J in JUDGES:
    if J["in_frame"]:
        ax.plot(J["col"], J["row"], "^", color="crimson", mec="k", ms=9)
        ax.annotate(f" {J['code']} ({J['s_g_km']:.1f} km)", (J["col"], J["row"]), fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8, label="distance from the mouth along the water, s (km)")
ax.set_title(f"{aoi.name}: the water-reachable frame and the judges", loc="left")
ax.set_axis_off();

## 2 · The record of each window

**Scenes.** The package's record is every scene of the Zarr stores with a catalogue instant and a
finite boundary level; clouds are masked pixel by pixel (SCL 4–7 clear). A window keeps the
scenes from its first day to the day after its last.

**Pixels.** `marea.extract`'s rule on the window's counts of clear (V) and wet (W) observations
under P0: a pixel with more than 30 clear observations is intertidal when its water frequency is
between 0.10 and 0.90, sea when it is at least 0.90. Its distance s from the mouth runs through
sea and intertidal pixels from the site's sea edge (`pyintertidal.geometry`); the pixels are
then grouped in 3-km bands of s holding at least 500 pixels each (`te.make_bands_fixed`).

In [ ]:
REC = {}
for w in WINDOWS:
    start, end = WINDOW_DATES[w]
    sel = np.flatnonzero((pkg.t >= start) & (pkg.t < end))             # the window's scenes
    clr, wet = (np.asarray(c, np.int64) for c in pkg.frame_counts(SETTING, w))
    with np.errstate(invalid="ignore", divide="ignore"):
        wf = np.where(clr > 30, wet / np.maximum(clr, 1), np.nan)      # water frequency
    sea = wf >= 0.90
    keep = np.intersect1d(np.flatnonzero((wf > 0.10) & (wf < 0.90)), pkg.dom)   # intertidal pixels
    inter = np.zeros(H * W, bool)
    inter[keep] = True
    seeds = geometry.mouth_seeds(sea, side=pkg.cfg.mouth_side)
    s_km = geometry.along_distance(sea | inter.reshape(H, W), seeds, pkg.pixel_m).ravel()[keep] / 1000
    with np.errstate(invalid="ignore"):          # isolated pixels (NaN s) go to band -1
        edges, centers, band_of = te.make_bands_fixed(s_km, 3.0, 500)
    REC[w] = dict(sel=sel, t=pkg.t[sel], keep=keep, pos=np.searchsorted(pkg.dom, keep),
                  s_km=s_km, band_of=band_of, centers=np.asarray(centers, float))
    print(f"{w}: {len(sel)} scenes ({pkg.t[sel][0]:%Y-%m-%d} .. {pkg.t[sel][-1]:%Y-%m-%d}), "
          f"{len(keep):,} intertidal px up to s = {np.nanmax(s_km):.1f} km, "
          f"{len(centers)} bands centred at {np.round(centers, 1).tolist()} km")


def observe(w, cols):
    """P0's observation of the intertidal pixels `cols` (sorted positions in keep) over the
    window's scenes: Y the NDWI, V valid (SCL 4-7 and a finite NDWI), W wet (V and NDWI > 0)."""
    return pkg.observe(SETTING, REC[w]["sel"], pos=REC[w]["pos"][cols])

## 3 · The boundary h_B

One provider, built as production builds it: `make_boundary` blends EOT20 with the nearest IOC
gauges at the mouth, the in-estuary gauges held out (EOT20 alone at Villaviciosa); its period is
the union of the stores, 2017–2025. Every method reads it the same way:

* `LEV`: h_B(t_i − τ) at the window's scene instants for every whole minute τ from −60 to 240,
  from one call over the shifted instants; its τ = 0 row must equal the record's h_B bit for bit;
* the 1-min series over the window ± 2 days, for the rates dh_B/dt, the high-water marks of its
  61-min running mean and the 10-min samples of the harmonic truth.

In [ ]:
pit.net.use_system_certificates()
t0 = time.time()
if S["n_gauges"]:
    boundary = make_boundary(aoi, tide_model="EOT20", n_gauges=S["n_gauges"],
                             period=("2017-01-01", "2025-12-31"), directory="tide_models",
                             cache_dir="data_v4/gauges/ioc_2017-2025", exclude=S["held_out"])
else:
    boundary = PyTMDBoundary("EOT20", *aoi.centroid, directory="tide_models")
print(f"h_B = {boundary.name} [{time.time() - t0:.0f} s]")

TAUS = truth.TAUS_A                               # -60 .. 240 min, every minute


def minutes(t):
    """Minutes since 1970-01-01 of naive-UTC instants."""
    return pd.DatetimeIndex(t).as_unit("ns").asi8 / 6e10


for w, r in REC.items():
    t0 = time.time()
    shifted = r["t"].values[None, :] - TAUS[:, None] * np.timedelta64(60, "s")     # t_i - tau
    inst = np.unique(shifted)
    lev = np.asarray(boundary.levels(pd.DatetimeIndex(inst)), float)
    r["LEV"] = lev[np.searchsorted(inst, shifted)]                                  # (n_tau, T)
    r["h0"] = r["LEV"][TAUS == 0][0]
    assert np.array_equal(r["h0"], pkg.h0[r["sel"]]), "h_B differs from the run's record"
    r["t_min"] = minutes(r["t"])
    start, end = (pd.Timestamp(d) for d in WINDOW_DATES[w])
    t1 = pd.date_range(start - pd.Timedelta(days=2), end + pd.Timedelta(days=2), freq="1min",
                       inclusive="left")
    r["t1"], r["h1"] = minutes(t1), np.asarray(boundary.levels(t1), float)
    r["marks"] = g21.tide_marks(r["t1"], levels.lowpass(r["h1"]))   # HW/LW of the 61-min mean
    on10 = (t1 >= start) & (t1 < end) & (t1.minute % 10 == 0)
    r["idx10"], r["h10"] = t1[on10], r["h1"][on10]
    print(f"{w}: h_B at {len(r['t'])} scenes x {len(TAUS)} lags, 1-min series of {len(t1):,} "
          f"samples, {int(r['marks']['is_hw'].sum())} high waters [{time.time() - t0:.0f} s]")


def h_B(r, t_min):
    """h_B at any instant (minutes), linear between the 1-min samples."""
    return np.interp(t_min, r["t1"], r["h1"], left=np.nan, right=np.nan)


def h_B_rows(r, taus):
    """Rows of LEV: h_B(t_i - tau) for whole-minute taus, shape (len(taus), T)."""
    return r["LEV"][np.searchsorted(TAUS, np.round(taus).astype(int))]


fig, axes = plt.subplots(len(REC), 1, figsize=(12, 2.6 * len(REC)), squeeze=False)
for ax, (w, r) in zip(axes[:, 0], REC.items()):
    ax.plot(r["t"], r["h0"], ".", ms=4, color="0.3")
    ax.set_ylabel("h_B (m)")
    ax.set_title(f"{w}: the {len(r['t'])} scenes at their boundary level", loc="left")
    ax.grid(alpha=0.3)
fig.tight_layout();

## 4 · The truth: the lag each judge shows

Each gauge is read as delivered (its stamps at the centre of its averaging window) and only
centred quantities are compared, because the datums differ. Per window and judge
(`experiments/benchmark/truth.py`, called directly):

* **(a) τ_a**: the τ that minimises E(τ), the median-centred RMS of h_B(t_i − τ) − g_i over the
  scene instants, g_i the gauge value nearest each instant within 6 min (`e_rows`,
  `best_lag`); its interval holds every τ within 1 cm of the minimum;
* **τ_ab**: the same with a free gain on h_B (`gain_aware`);
* **(b) τ_b**: the M2 phase of the gauge minus that of h_B, on their common 10-min samples
  (52 constituents, `m2_lag`), with a jackknife over months (one-year windows) or quarters.

Positive τ: the local water is later than h_B.

In [ ]:
TRUTH, CURVES = {}, {}
for w, r in REC.items():
    start, end = (pd.Timestamp(d) for d in WINDOW_DATES[w])
    group = "month" if end.year - start.year == 1 else "quarter"
    for J in JUDGES:
        code = J["code"]
        series = judges.centred(judges.read_series(J), J)        # stamps at the window centre
        inc = judges.window_inclusion(J, series, r["t"], w)
        if not inc["included"]:
            print(f"{w} {code}: excluded ({', '.join(inc['exclusion'])})")
            continue
        g = judges.at_instants(series, r["t"], truth.TOL_MIN)   # gauge at the scene instants
        E = truth.e_rows(r["LEV"], g)
        a = truth.best_lag(TAUS, E)
        E_ab, gain = truth.gain_aware(r["LEV"], g)
        ab = truth.best_lag(TAUS, E_ab)
        g10 = judges.centred_10min(series, J["sampling_min"], grid=r["idx10"]).to_numpy(float)
        b = truth.m2_lag(r["idx10"], r["h10"], g10, group=group)
        TRUTH[w, code] = {"a": a["tau"], "ab": ab["tau"], "b": b["tau"]}
        CURVES[w, code] = E
        print(f"{w} {code:28s} s_g {J['s_g_km']:5.1f} km, {int(np.isfinite(g).sum()):3d} instants: "
              f"tau_a {a['tau']:5.0f} [{a['lo']:.0f}, {a['hi']:.0f}], tau_ab {ab['tau']:5.0f} "
              f"(gain {gain[np.nanargmin(E_ab)]:.2f}), tau_b {b['tau']:6.1f} ± {1.96 * b['se']:.1f} min; "
              f"E(0) {100 * E[TAUS == 0][0]:.1f} cm, E(tau_a) {100 * a['Emin_m']:.1f} cm")

In [ ]:
if CURVES:
    fig, axes = plt.subplots(1, len(REC), figsize=(6 * len(REC), 4), squeeze=False)
    for ax, w in zip(axes[0], REC):
        for (ww, code), E in CURVES.items():
            if ww == w:
                ax.plot(TAUS, 100 * E, label=f"{code} (tau_a {TRUTH[w, code]['a']:.0f} min)")
        ax.set_xlabel("lag tau (min)")
        ax.set_ylabel("E(tau): misfit of h_B(t - tau) at the gauge (cm)")
        ax.set_title(w, loc="left")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()

## 5 · MAREA's band lags (L01) and the lag at each judge

MAREA's lag stage exactly as `marea.reconstruct` runs it: at most 1 200 pixels per band (the
seeded draw of `configs/m2.yaml`), one lag per band by profiled Bernoulli likelihood of their
wet/dry record against the shifted boundary (`te.m2a_rasch`, monotone inland, the shift bank of
`marea.BANK_TAUS`), a negative lag set to 0 (causality), and the lag applied to each pixel: the
band lags smoothed along s with a 3-km kernel and rounded to the minute.

**Read-out at a judge** (the scorer's rule, `experiments/benchmark/readout.py`, one rule for
every method): the median of the method's lag over the judge's neighbourhood when at least 30
pixels there have one (step 1); otherwise the median over the judge's band, the band of most of
its 200 nearest intertidal pixels (step 2); otherwise 0 (step 3). Phase-type lags (GM20, HD26-H,
PPL) take a circular median.

In [ ]:
for w, r in REC.items():
    r["judges"] = [J for J in JUDGES if (w, J["code"]) in TRUTH]              # scored here
    r["nbhd"] = {J["code"]: readout.nbhd_positions(J["nbhd"], r["keep"]) for J in r["judges"]}
    r["k_g"] = {J["code"]: judges.station_geometry(J["lon"], J["lat"], grid, r["keep"], r["s_km"],
                                                   r["band_of"])["k_g"] for J in r["judges"]}


def read_out(w, J, lag_at, phase=False):
    """(lag, step) of a method at judge J; lag_at(cols) gives its lags at keep positions cols."""
    r, code = REC[w], J["code"]
    median = common.circ_median if phase else np.median
    v = np.asarray(lag_at(r["nbhd"][code]), float)
    v = v[np.isfinite(v)]
    if len(v) >= 30:
        return float(median(v)), 1
    k = r["k_g"][code]
    v = np.asarray(lag_at(np.flatnonzero(r["band_of"] == k)), float) if k >= 0 else np.zeros(0)
    v = v[np.isfinite(v)]
    if len(v):
        return float(median(v)), 2
    return 0.0, 3


TRUTHS = ("a", "ab", "b")
LAG, BAND = [], {}             # read-outs at the judges; band lags of the band methods


def put(method, w, code, tau, step):
    """One read-out: one lag for the three truths, or a dict {truth: lag} (point rows)."""
    taus = tau if isinstance(tau, dict) else {T: tau for T in TRUTHS}
    LAG.append(dict(window=w, judge=code, method=method, step=step, **taus))

In [ ]:
M2 = yaml.safe_load(open("configs/m2.yaml", encoding="utf-8"))
SEED = M2["seed"] + 31                            # marea.reconstruct's column seed
MAX_PX = M2["max_px_banda"]["m2a"]
FIT_KW = dict(sigma0=M2["sigma0_m"], sg_grid=tuple(M2["sigma_perfil_m"]), tau_bounds=marea.TAU_BOUNDS,
              z_points=M2["z_puntos"], max_px_band=MAX_PX, monotone=True, fit_alpha=False)

for w, r in REC.items():
    t0 = time.time()
    K = len(r["centers"])
    rng = np.random.default_rng(SEED)
    cols = []
    for k in range(K):                            # the seeded draw of at most 1 200 px per band
        c = np.flatnonzero(r["band_of"] == k)
        cols.append(np.sort(rng.choice(c, MAX_PX, replace=False)) if len(c) > MAX_PX else c)
    cols = np.sort(np.concatenate(cols))
    o = observe(w, cols)
    bank = te.ShiftBank(list(marea.BANK_TAUS), list(h_B_rows(r, marea.BANK_TAUS)))
    fit = te.m2a_rasch(o.W, o.V, bank, r["band_of"][cols], r["centers"], K,
                       rng=np.random.default_rng(SEED), **FIT_KW)
    tau = np.asarray(fit["tau"], float)
    tau = np.where(np.abs(tau) > marea.TAU_DETECT_MIN, tau, 0.0)
    tau = np.where(tau < 0, 0.0, tau)              # causality: a delay is not negative
    tau_px = te.smooth_lag_profile(r["centers"], tau, r["s_km"], bandwidth_km=3.0)
    tau_px = np.where(np.isfinite(tau_px), np.round(tau_px), 0.0)   # the lag each pixel receives
    BAND["L01", w] = tau
    print(f"{w}: m2a on {len(cols):,} px of {K} bands -> band lags {np.round(tau, 1).tolist()} min "
          f"[{time.time() - t0:.0f} s]")
    for J in r["judges"]:
        lag, step = read_out(w, J, lambda c: tau_px[c])
        put("L01", w, J["code"], lag, step)
        print(f"    {J['code']:28s} s_g {J['s_g_km']:5.1f} km: MAREA {lag:5.1f} min (step {step}), "
              f"tau_a {TRUTH[w, J['code']]['a']:.0f} min")

## 6 · Zero lag (L17)

The boundary read at the scene instant itself: τ = 0 at every judge.

In [ ]:
for w, r in REC.items():
    for J in r["judges"]:
        put("L17", w, J["code"], 0.0, "point")
print(f"{sum(len(r['judges']) for r in REC.values())} judge-windows at 0 min")

## 7 · The global tide model at the gauges (L14)

The run's primary global model, EOT20, predicted at each gauge's position (pyTMD's nearest ocean
cell) and put through the truth functions in the gauge's place: the lag of the model against
h_B by (a), τ_ab and (b). It is what a user without imagery would apply; the nearest ocean
cell of the 1/8° model can lie 10–30 km from an inner gauge, so it is a lower bound.

In [ ]:
for w, r in REC.items():
    for J in r["judges"]:
        model = PyTMDBoundary(GLOBAL_MODEL, J["lat"], J["lon"], directory="tide_models")
        m = np.asarray(model.levels(r["t"]), float)
        a = truth.best_lag(TAUS, truth.e_rows(r["LEV"], m))
        ab = truth.best_lag(TAUS, truth.gain_aware(r["LEV"], m)[0])
        b = truth.m2_lag(r["idx10"], r["h10"], np.asarray(model.levels(r["idx10"]), float))
        put("L14", w, J["code"], {"a": a["tau"], "ab": ab["tau"], "b": b["tau"]}, "point")
        print(f"{w} {J['code']:28s} {GLOBAL_MODEL}: (a) {a['tau']:.0f}, tau_ab {ab['tau']:.0f}, "
              f"(b) {b['tau']:.1f} min")

## 8 · The imagery competitors

Every method reads the same scenes, pixels and boundary rows. Five work on aggregates of each
3-km band, the share of its valid pixels that are wet and the median index, on the scenes where
at least 80 % of the band is valid (`experiments/benchmark/lags/aggregate.py`, `hd26.py`):

* **L04 XC** (wet share L04w, median index L04i): the lag of the best Pearson correlation with
  h_B(t − τ), τ from −45 to 180 min every 5 (`aggregate.xc_band`);
* **L07 LOOP**: the lag at which the rising- and falling-tide scenes fall on one increasing
  curve of the wet share against h_B(t − τ) (`aggregate.loop_band`);
* **L09 HD26-H** (Hart-Davis et al. 2026): the M2 phase of the band's wet share minus that of
  h_B at the same scenes; it needs 365 days between first and last scene (`hd26.l09_band`);
* **L10 HD26-R2**: a delayed and smeared h_B with an isotonic link, at least 60 scenes
  (`aggregate.l10_band`).

First the band aggregates, one band read at a time.

In [ ]:
AGG = {}
for w, r in REC.items():
    t0 = time.time()
    T, K = len(r["sel"]), len(r["centers"])
    n_px = np.bincount(r["band_of"][r["band_of"] >= 0], minlength=K)
    n_valid, n_wet = np.zeros((T, K), np.int64), np.zeros((T, K), np.int64)
    median_Y = np.full((T, K), np.nan, np.float32)
    for k in range(K):
        o = observe(w, np.flatnonzero(r["band_of"] == k))
        n_valid[:, k] = o.V.sum(axis=1)
        n_wet[:, k] = (o.W & o.V).sum(axis=1)
        for i in range(T):
            y = o.Y[i][o.V[i]]
            if y.size:
                median_Y[i, k] = np.median(y)
        del o
    with np.errstate(invalid="ignore", divide="ignore"):
        AGG[w] = dict(cover=n_valid / n_px, wet=np.where(n_valid > 0, n_wet / np.maximum(n_valid, 1), np.nan),
                      index=median_Y)
    print(f"{w}: {K} bands of {n_px.min():,}-{n_px.max():,} px x {T} scenes [{time.time() - t0:.0f} s]")

In [ ]:
S_GRID = [0.0, 15.0, 30.0, 60.0]                    # L10's smearing widths (min)


def smeared(r, s):
    """sum_m w_m h_B(t - tau - m): h_B delayed by tau and smeared by a Gaussian of width s (min)."""
    if s == 0:
        return h_B_rows(r, common.TAU_SCAN)
    m, wts = aggregate.gaussian_weights(s)
    return np.array([wts @ h_B(r, r["t_min"][None, :] - tau - m[:, None].astype(float))
                     for tau in common.TAU_SCAN])


for w, r in REC.items():
    t0 = time.time()
    A, K = AGG[w], len(r["centers"])
    H_scan = h_B_rows(r, common.TAU_SCAN)                       # h_B(t - tau), tau = -45..180 by 5
    R_scan = np.array([(h_B(r, r["t_min"] - tau + 1) - h_B(r, r["t_min"] - tau - 1)) / 2
                       for tau in common.TAU_SCAN])            # dh_B/dt at t - tau (m/min)
    U_scan = [smeared(r, s) for s in S_GRID]
    t_days = truth.days_since_1992(r["t"])
    for pid in ("L04w", "L04i", "L07", "L09", "L10"):
        proxy = A["index"].astype(float) if pid == "L04i" else A["wet"]
        tau = np.full(K, np.nan)
        for k in range(K):
            use = (A["cover"][:, k] >= 0.8) & np.isfinite(proxy[:, k])
            y = proxy[use, k]
            if pid in ("L04w", "L04i"):
                tau[k] = aggregate.xc_band(y, H_scan[:, use])["tau"]
            elif pid == "L07":
                tau[k] = aggregate.loop_band(y, H_scan[:, use], R_scan[:, use])["tau"]
            elif pid == "L09":
                tau[k] = hd26.l09_band(t_days[use], y, r["h0"][use])["tau"]
            else:
                tau[k] = aggregate.l10_band(y, [U[:, use] for U in U_scan], S_GRID)["tau"]
        if not np.isfinite(tau).any():
            print(f"{w} {pid}: no band identified (NOT COMPUTABLE)")
            continue
        BAND[pid, w] = tau
        field = common.broadcast_band(tau, r["band_of"])        # the band lag on its pixels
        for J in r["judges"]:
            put(pid, w, J["code"], *read_out(w, J, lambda c: field[c], phase=(pid == "L09")))
        print(f"{w} {pid}: band lags {np.round(tau, 1).tolist()} min")
    print(f"{w} [{time.time() - t0:.0f} s]")

Three methods give one lag per pixel; the notebook computes them only where a judge reads them
(its neighbourhood, and its band when the neighbourhood has fewer than 30 defined pixels), in
blocks of 5 000 pixels. Each pixel is fitted alone, so the lags are those of the whole field.

* **L08 GM20** (Geyman & Maloof 2020): each pixel's index folded on the hours since h_B's last
  high water and fitted with one M2 sinusoid; the lag is the phase of its crest minus that of
  h_B on the same scenes (`gm20.fold_fit`; QC: ≥ 20 scenes, SD > 0.2, wet share 0.05–0.95,
  R² ≥ 0.1);
* **L11 PPL**: a logistic regression of wet/dry on h_B and its rate, the boundary's phase
  plane; the angle of the wet/dry line is the lag (`ppl.logistic_fit`; QC: ≥ 20 scenes, wet
  share 0.05–0.95, converged, wet at high water, likelihood-ratio p < 0.01);
* **L12 BT26-shift** (after Bishop-Taylor et al. 2026): the τ that maximises the correlation of
  the pixel's index with h_B(t − τ) (`bt26.corr_block`; QC: ≥ 20 scenes, r ≥ 0.3).

In [ ]:
def gm20_lags(w, o):
    r = REC[w]
    C, x_ok = gm20.fold_design(gm20.hours_since_hw(r["t_min"], r["marks"]))
    fit = gm20.fold_fit(C, o.Y, o.V & x_ok[:, None], h0=r["h0"], W=o.W)
    good = np.logical_and.reduce(list(gm20.qc_mask(fit).values()))
    return np.where(good, truth.wrap_gm20(fit["phase_y"] - fit["phase_h"]), np.nan)


def ppl_lags(w, o):
    r = REC[w]
    x1 = r["h0"] - r["h0"].mean()                                 # h_B, centred
    x2 = (h_B(r, r["t_min"] + 1) - h_B(r, r["t_min"] - 1)) / 2 / common.OMEGA_M2_PER_MIN
    ok = np.isfinite(x1) & np.isfinite(x2)
    X = np.column_stack([np.ones_like(x1), np.where(ok, x1, 0.0), np.where(ok, x2, 0.0)])
    X[~ok] = 0.0                                                  # a scene without a rate is left out
    fit = ppl.logistic_fit(X, o.W, o.V & ok[:, None])
    good = np.logical_and.reduce(list(ppl.qc_mask(fit)[0].values()))
    return np.where(good, ppl.tau_of(fit["beta"]), np.nan)


def bt26_lags(w, o):
    H_scan = h_B_rows(REC[w], common.TAU_SCAN)
    r_, n = bt26.corr_block(o.Y, o.V, H_scan - H_scan.mean(axis=1, keepdims=True))
    fin = np.isfinite(r_)
    r_max = np.where(fin.any(axis=1), np.max(np.where(fin, r_, -np.inf), axis=1), np.nan)
    tau, _ = bt26.vertex_rows(r_)
    with np.errstate(invalid="ignore"):
        return np.where((n >= 20) & (r_max >= 0.3) & np.isfinite(tau), tau, np.nan)


def by_blocks(w, cols, lags, size=5000):
    """A per-pixel method at keep positions cols, read in blocks of `size` pixels."""
    out = np.full(len(cols), np.nan)
    for a in range(0, len(cols), size):
        out[a:a + size] = lags(w, observe(w, cols[a:a + size]))
    return out


for pid, lags, phase in (("L08", gm20_lags, True), ("L11", ppl_lags, True), ("L12", bt26_lags, False)):
    for w, r in REC.items():
        t0 = time.time()
        for J in r["judges"]:
            lag, step = read_out(w, J, lambda c: by_blocks(w, c, lags), phase=phase)
            put(pid, w, J["code"], lag, step)
            print(f"{w} {pid} {J['code']:28s} {lag:6.1f} min (step {step})")
        print(f"{w} {pid} [{time.time() - t0:.0f} s]")

## 9 · Physics and reference rows (L18, L20)

* **L18 shallow-water celerity**: the travel time of a long wave up the channel,
  τ(s) = ∫ ds' / √(g h̄(s')), with h̄ the mean depth below mean sea level of the permanently wet
  channel (water frequency ≥ 0.98) per 1-km bin of s, from the EMODnet DTM 2024 (referred to
  LAT; LAT taken as the window's lowest h_B, the depth floored at 1 m; `physics.bin_depths`,
  `physics.tau_of_s`);
* **L20 reference**: the judges' own truths interpolated linearly in s, each location left out,
  anchored at τ = 0 at the mouth: what a gauge network would give. Never ranked.

In [ ]:
if JUDGES:
    geo = pkg.geometry
    channel = np.asarray(geo["wf"], float).reshape(H, W) >= 0.98            # permanently wet
    dtm, _ = emodnet.verified_dtm(f"{V2_ROOT}/common/{SITE}/emodnet")
    elev = emodnet.dtm_on_grid(dtm, grid).astype(float)                     # m above LAT
    s_g_all = [J["s_g_km"] for J in JUDGES if J["in_frame"]]
    for w, r in REC.items():
        start, end = (minutes([d])[0] for d in WINDOW_DATES[w])
        h = r["h1"][(r["t1"] >= start) & (r["t1"] < end) & np.isfinite(r["h1"])]
        depth = (h.mean() - h.min()) - elev                                   # below mean sea level
        chan = channel & np.isfinite(s_union) & np.isfinite(depth)
        s_max = np.nanmax(np.concatenate([s_union[chan], r["s_km"][np.isfinite(r["s_km"])], s_g_all]))
        hbar = physics.bin_depths(s_union[chan], depth[chan], int(s_max) + 1)["hbar"]
        for J in r["judges"]:
            put("L18", w, J["code"], float(physics.tau_of_s(np.array([J["s_g_km"]]), hbar)[0]), "point")
        print(f"{w}: MSL - LAT {h.mean() - h.min():.2f} m; channel depth by km: {np.round(hbar, 1).tolist()}")

for w, r in REC.items():
    loc_s, loc_tau = {}, {}
    for J in r["judges"]:                         # the Terneuzen pair is one location
        loc_s.setdefault(J["location"], []).append(J["s_g_km"])
        loc_tau.setdefault(J["location"], []).append(TRUTH[w, J["code"]])
    for J in r["judges"]:
        ref = {}
        for T in TRUTHS:
            nodes = sorted([(0.0, 0.0)] + [(np.mean(loc_s[L]), np.mean([x[T] for x in loc_tau[L]]))
                                           for L in loc_s if L != J["location"]])
            ref[T] = float(np.interp(np.mean(loc_s[J["location"]]), *zip(*nodes)))
        put("L20", w, J["code"], ref, "point")

## 10 · Errors per judge and per method

e = τ_method − τ_truth, in minutes, circular (one M2 period) for the phase-type lags: GM20,
HD26-H, PPL and the global model's (b). Per window, method and truth: the gauges of one location
are averaged first (the two Terneuzen gauges), then every location counts once: MAE = mean |e|,
bias = mean e, hit10 = share of locations within 10 min. The per-judge rows go to
`<OUT_DIR>/<site>.csv` (Villaviciosa: its band lags).

In [ ]:
NAMES = {"L01": "MAREA (m2a band lags)", "L17": "zero lag", "L14": f"{GLOBAL_MODEL} at the gauge",
         "L04w": "XC wet share", "L04i": "XC median index", "L07": "LOOP", "L08": "GM20",
         "L09": "HD26-H", "L10": "HD26-R2", "L11": "PPL", "L12": "BT26-shift",
         "L18": "shallow-water celerity", "L20": "REF gauge network (never ranked)"}
PHASE = {"L08", "L09", "L11"}
JUDGE = {J["code"]: J for J in JUDGES}

rows = []
for x in LAG:
    J = JUDGE[x["judge"]]
    for T in TRUTHS:
        tv = TRUTH[x["window"], x["judge"]][T]
        phase = x["method"] in PHASE or (x["method"] == "L14" and T == "b")
        e = float(common.circ_diff(x[T], tv)) if phase else x[T] - tv
        rows.append(dict(site=SITE, window=x["window"], judge=x["judge"], location=J["location"],
                         s_g_km=J["s_g_km"], method=x["method"], truth=T, tau_m=x[T],
                         tau_true=tv, e=e, readout_step=x["step"]))
points = pd.DataFrame(rows, columns=["site", "window", "judge", "location", "s_g_km", "method",
                                     "truth", "tau_m", "tau_true", "e", "readout_step"])


def site_errors(points):
    """MAE, bias and hit10 per window, method and truth: locations first, each counted once."""
    loc = (points.assign(abs_e=points.e.abs())
           .groupby(["window", "method", "truth", "location"])[["e", "abs_e"]].mean().reset_index())
    return (loc.groupby(["window", "method", "truth"])
            .agg(n_loc=("location", "size"), MAE=("abs_e", "mean"), bias=("e", "mean"),
                 hit10=("abs_e", lambda v: 100 * np.mean(v <= 10))).reset_index())


if JUDGES:
    points.to_csv(f"{OUT_DIR}/{SITE}.csv", index=False)
    errors = site_errors(points)
    for w in WINDOWS:
        print(f"\n{SITE} {w}: e = method - truth (a), minutes")
        print(points[(points.window == w) & (points.truth == "a")]
              .pivot_table(index="method", columns="judge", values="e", sort=False).round(1).to_string())
    table = errors.pivot_table(index=["window", "method"], columns="truth", values=["MAE", "bias", "hit10"])
    table = table.reindex([(w, m) for w in WINDOWS for m in NAMES if (w, m) in table.index])
    display(table.round(1))
else:
    pd.DataFrame([dict(site=SITE, window=w, method=m, band=k, centre_km=REC[w]["centers"][k], tau=v)
                  for (m, w), tau in BAND.items() for k, v in enumerate(tau)]
                 ).to_csv(f"{OUT_DIR}/{SITE}.csv", index=False)
    print(f"{SITE} has no judge: its band lags -> {OUT_DIR}/{SITE}.csv")

## 11 · Figure

Left: the lag along the estuary, the band methods at their band centres (the phase-type HD26-H
wrapped about 67.5 min), the judges' truths and the other methods' read-outs at the judges.
Right: MAE against truths (a) and (b) per method.

In [ ]:
COLORS = {"L01": "#0A7FA8", "L04w": "#7FB800", "L04i": "#4D7300", "L07": "#9B59B6", "L09": "#E67E22",
          "L10": "#C0392B", "L08": "#F1C40F", "L11": "#16A085", "L12": "#2C3E50", "L14": "#B7791F",
          "L17": "0.6", "L18": "#8E44AD", "L20": "0.3"}
fig, axes = plt.subplots(len(REC), 2, figsize=(15, 5 * len(REC)), squeeze=False,
                         gridspec_kw={"width_ratios": [3, 2]})
for (axl, axr), (w, r) in zip(axes, REC.items()):
    for (m, ww), tau in BAND.items():
        if ww == w:
            y = common.wrap_phase(tau) if m in PHASE else tau
            axl.plot(r["centers"], y, "-o", color=COLORS[m], lw=3 if m == "L01" else 1, ms=4,
                     label=NAMES[m], zorder=3 if m == "L01" else 2)
    pw = points[(points.window == w) & (points.truth == "a")]
    for i, m in enumerate(["L08", "L11", "L12", "L14", "L18", "L20"]):
        q = pw[pw.method == m]
        axl.plot(q.s_g_km + 0.15 * (i - 2.5), q.tau_m, "D", ms=6, color=COLORS[m], label=NAMES[m] if len(q) else None)
    for J in r["judges"]:
        tv = TRUTH[w, J["code"]]
        axl.plot(J["s_g_km"], tv["a"], "*", ms=16, color="k", label="truth (a)" if J is r["judges"][0] else None)
        axl.plot(J["s_g_km"], tv["b"], "o", ms=11, mfc="none", mec="k", label="truth (b)" if J is r["judges"][0] else None)
    axl.axhline(0, color="0.5", lw=0.8)
    axl.set_xlabel("distance from the mouth, s (km)")
    axl.set_ylabel("lag behind h_B (min)")
    axl.set_title(f"{aoi.name} · {w}", loc="left")
    axl.grid(alpha=0.3)
    axl.legend(fontsize=7, ncol=2)
    if JUDGES:
        ew = errors[errors.window == w].pivot(index="method", columns="truth", values="MAE")
        ew = ew.reindex([m for m in NAMES if m in ew.index])
        yy = np.arange(len(ew))
        axr.barh(yy - 0.2, ew["a"], height=0.4, color="#0A7FA8", label="truth (a)")
        axr.barh(yy + 0.2, ew["b"], height=0.4, color="#B7791F", label="truth (b)")
        axr.set_yticks(yy, [NAMES[m] for m in ew.index], fontsize=8)
        axr.invert_yaxis()
        axr.set_xlabel("MAE over the locations (min)")
        axr.grid(alpha=0.3, axis="x")
        axr.legend(fontsize=8)
    else:
        axr.set_axis_off()
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/{SITE}.png", dpi=130, bbox_inches="tight")

## 12 · Year-to-year coherence of the band lags

A truth-free check where a site has two or more calendar-year windows (W1 is 2025): each band
method's lags per 3-km bin of the band centre, year by year, and the median |difference| over
the pairs of years.

In [ ]:
YEAR = {"Y2023": "2023", "Y2024": "2024", "W1": "2025"}
years = [w for w in WINDOWS if w in YEAR]
coherence = pd.DataFrame()
if len(years) >= 2:
    rows = []
    for m in dict.fromkeys(m for m, _ in BAND):
        by_bin = {}
        for w in years:
            for c, v in zip(REC[w]["centers"], BAND.get((m, w), np.full(len(REC[w]["centers"]), np.nan))):
                by_bin.setdefault(int(c // 3.0), {})[YEAR[w]] = v
        for b, vals in sorted(by_bin.items()):
            v = [x for x in vals.values() if np.isfinite(x)]
            d = [abs(float(common.circ_diff(p, q)) if m in PHASE else p - q)
                 for i, p in enumerate(v) for q in v[i + 1:]]
            rows.append(dict(method=m, band=b, **{YEAR[w]: vals.get(YEAR[w], np.nan) for w in years},
                             median_abs_diff=np.median(d) if d else np.nan))
    coherence = pd.DataFrame(rows)
    display(coherence.round(1))
    print(coherence.groupby("method", sort=False).median_abs_diff.median().round(1).to_string())
else:
    print("one calendar year only: no year-to-year comparison here")

## 13 · Check against the server run

The run's `lag_points.csv` (rows with status ok, the same windows, methods and truths),
summarised with the same rule, must give the same MAE and bias within 0.01 min. Where section 12
ran, its band lags must equal the run's `lag_repro.csv` (written to 2 decimals).

In [ ]:
if JUDGES:
    run = pd.read_csv(f"{SCORES}/lag_points.csv", comment="#")
    run = run[(run.site == SITE) & run.window.isin(WINDOWS) & run.truth.isin(TRUTHS)
              & run["product"].isin(list(NAMES)) & (run.status == "ok")]
    run = run.drop(columns="method").rename(columns={"product": "method"})   # its 'method' is the unit
    both = site_errors(points).merge(site_errors(run), on=["window", "method", "truth"], how="outer",
                                     suffixes=("", "_run"), indicator=True)
    worst = {c: float((both[c] - both[f"{c}_run"]).abs().max()) for c in ("MAE", "bias")}
    print(f"{len(both)} (window, method, truth) rows; in one run only: "
          f"{int((both._merge != 'both').sum())}; largest differences (min): {worst}")
    display(both[both.truth == "a"][["window", "method", "n_loc", "MAE", "MAE_run", "bias", "bias_run"]].round(2))
    assert (both._merge == "both").all() and max(worst.values()) <= 0.01

if len(coherence):
    rep = pd.read_csv(f"{SCORES}/lag_repro.csv", comment="#")
    rep = rep[(rep.site == SITE) & (rep.kind == "years") & (rep.band != "all")]
    diffs = []
    for _, x in rep.iterrows():
        mine = coherence[(coherence.method == x["product"]) & (coherence.band == int(x["band"]))]
        for yr, v in zip(x["groups"].split(";"), x["values"].split(";")):
            if len(mine):
                a, b = float(mine[yr[1:]].iloc[0]), float(v)              # yr: "Y2023" -> "2023"
                diffs.append(np.nan_to_num(abs(a - b)) if np.isnan(a) == np.isnan(b) else np.inf)
    print(f"{len(diffs)} band lags compared with lag_repro.csv; largest difference {max(diffs):.3f} min")
    assert diffs and max(diffs) <= 0.006